# Visualize Model Predictions

This notebook loads a trained model and visualizes its predictions on a new sample.

In [ ]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path
import subprocess
from sklearn.manifold import TSNE


project_root = Path.cwd().parent.parent
project_branch = 'modis_himalaya'
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))


from plot_utilities import *

# Import utilities
from utilities import KernelNN, KernelNN_subgraph, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
import time
from IPython.display import display
import plotly.express as px

In [ ]:
def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / project_branch / 'trained_models' / model_type / id
    with open(model_path, 'r') as f:
        config = json.load(f)
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    return model
    

In [ ]:
# choose data
model_type = 'regular'


# ---------------------------------------------------------------------------
sys.path.append(f"{project_root}/{project_branch}/scripts/")
from utilities_modis_himalaya import MODIS_Himalaya_Graph_Generator as Data_Generator



# Paths
trained_models_path = project_root / project_branch / 'trained_models' / model_type
training_logs_path = trained_models_path / 'logs'
data_path = project_root / project_branch / 'data'  # Adjust if needed

print('Available models:\n')
files = sorted(os.listdir(trained_models_path), reverse=True)

for i, f in enumerate(files):
    if (Path(f).suffix == ".pth"):
        print(f'[{i}] {f}')


## 2. Load Model and Configuration

In [ ]:
# Set model ID
model_id = 'testtraining2_2026-05-05_t08-38-27'
# model_id = files[19]#'reg_medium_woPos_deep6-regular-2026-04-13_t18-58-47'



# ---------------------------------------------------------------------------
if model_id.endswith('.pth'):
    model_id = model_id[:-4]

# Load full configuration (includes model, dataset, and training configs)
config_path = f'{training_logs_path}/configs/{model_id}.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# Load loss records
loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')

print('Configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')

print(f'\Training completed {len(loss_df)} epochs')
print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}')
print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(loss_df['epoch'], loss_df['loss'], label='Training Loss', marker='o')
ax.plot(loss_df['epoch'], loss_df['val_loss'], label='Validation Loss', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Training History - {model_id}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
model = KernelNN(
    width=config['width'],
    ker_width=config['ker_width'],
    depth=config['depth'],
    ker_in=config['edge_attr'],
    in_width=config['node_features'],
    out_width=config['out_width']
).to(device)

# Load trained weights
model_path = f'{trained_models_path}/{model_id}.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'Model loaded: {model_id}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Create New Sample

Use the same functionality as dataset creation to generate a single sample.

In [ ]:
config["bbox"]

In [ ]:
#brighttemp_1KM_grid_modis_20170101_0350
timestamp = "20180102_0540"

N=500
k=config["k"]
perplexity=config["perplexity"]
sampling_method=config["sampling_method"]
feature_mode=config["feature_mode"]
bbox=[500, 500, 1500, 1500]#config["bbox"]

Generator = Data_Generator(f'{data_path}/himalaya/')
Generator.load_data(timestamp, bbox=bbox)
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')


print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})

In [ ]:
print('predict results')
output = model(sample).detach().cpu().numpy()

---
Plot

In [ ]:
locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})

res_model = pd.DataFrame(
    {
        "x1": output[:, 0],
        "x2": output[:, 1],
        "x3": output[:, 2],
    }
)
res_model = pd.concat([locations, res_model], axis=1)

figsize = (5, 5)
fig = rgb(res_model, figsize=figsize)
#fig = rgb_scale(res_model, percentile=1, figsize=figsize)

In [ ]:
fig_3d = px.scatter_3d(
        res_model,
        x="x1",
        y="x2",
        z="x3",
        color="x3",  # optional coloring
        title=f"3D Scatter: {model_id}"
    )
fig_3d.update_traces(marker=dict(size=1)) 
fig_3d.show()

In [ ]:
X = sample.x.detach().cpu().numpy() # shape: (samples, features)
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

#locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})

In [ ]:
# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations, res_pca], axis=1)

fig = rgb(res_pca, figsize=figsize)

In [ ]:
fig_3d = px.scatter_3d(
        res_pca,
        x="x1",
        y="x2",
        z="x3",
        color="x3",  # optional coloring
        title=f"3D Scatter: PCA"
    )
fig_3d.update_traces(marker=dict(size=1)) 
fig_3d.show()

In [ ]:

# -------------------------
# t-SNE
# -------------------------
X = sample.normed_values.detach().cpu().numpy()

tsne = TSNE(
    n_components=3,
    perplexity=5,
    learning_rate=200,
    random_state=42
)

output = tsne.fit_transform(X)


res_tsne= pd.DataFrame(output, columns=["x1", "x2", "x3"])
res_tsne = pd.concat([locations, output], axis=1)

fig = rgb(res_tsne, figsize=figsize)

In [ ]:
# UMAP
dir = f'{project_root}/ERA5/umap'


X = sample.normed_values.detach().cpu().numpy()

umap_id = f'umap_buffer_{i}'
np.save(f'{dir}/{umap_id}.npy', X)

subprocess.run(
    f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}",
    shell=True
)

pred_umap = np.load(f'{dir}/res_{umap_id}.npy')

os.remove(f'{dir}/{umap_id}.npy')
os.remove(f'{dir}/res_{umap_id}.npy')

In [ ]:
# create pandas DataFrame
res_umap= pd.DataFrame(pred_umap, columns=["x1", "x2", "x3"])
res_umap = pd.concat([locations, res_umap], axis=1)

fig = rgb(res_umap, figsize=figsize)